# 15 — Conversation Memory (`memory/checkpointer.py`)
LangGraph conversation memory via SQLite (dev) or PostgreSQL (prod).

The `get_checkpointer()` function returns either:
- `SqliteSaver` (default, dev) — file-based, zero setup, stored in `data/memory.db`
- `PostgresSaver` (prod) — shared across ECS tasks

Memory is passed as `checkpointer=` to `workflow.compile()`. Each `thread_id` maintains its own conversation history.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'codebase', 'codebase', 'src'))
os.makedirs("logs", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.environ["ENVIRONMENT"] = "development"
os.environ["SQLITE_PATH"] = "./data/notebook_memory.db" 

## 1. get_checkpointer — Returns SqliteSaver in Dev

In [ ]:
from memory.checkpointer import get_checkpointer
from langgraph.checkpoint.sqlite import SqliteSaver

checkpointer = get_checkpointer()
print("Type:", type(checkpointer).__name__)
print("Is SqliteSaver:", isinstance(checkpointer, SqliteSaver))

## 2. Conversation History Accumulation (operator.add)

In [ ]:
import operator

# conversation_history uses Annotated[List[dict], operator.add]
# Each synthesizer call appends ONE entry — it never overwrites

entry1 = [{"query": "What is GRR?", "summary": "GRR is 87.5%", "intent": "knowledge_lookup", "tickets": []}]
entry2 = [{"query": "Why did it drop?", "summary": "Churn increased", "intent": "full_diagnostic", "tickets": ["DATA-001"]}]

accumulated = operator.add(entry1, entry2)
print(f"After 2 turns: {len(accumulated)} entries")
for e in accumulated:
    print(f"  - {e['query'][:40]} → {e['summary'][:40]}")

## 3. Thread-Based Memory Isolation

In [ ]:
from graph.state import initial_state

# Each thread_id gets its own isolated state
thread_a = initial_state("What is GRR?", thread_id="thread-alice", user_id="alice")
thread_b = initial_state("What is CAC?", thread_id="thread-bob",   user_id="bob")

print("Thread A:", thread_a["thread_id"], "| query:", thread_a["query"])
print("Thread B:", thread_b["thread_id"], "| query:", thread_b["query"])
print("Histories start empty:", thread_a["conversation_history"])

## 4. user_preferences — Persistent User Settings

In [ ]:
# user_preferences is a plain dict (not accumulated) — last write wins
# Useful for storing user's preferred time_range, products, notification settings

prefs_example = {
    "preferred_time_range": "last_quarter",
    "default_products": ["retention", "bookings"],
    "notification_email": "analyst@company.com",
}

state = initial_state("show me metrics")
state["user_preferences"] = prefs_example
print("User preferences:", state["user_preferences"])

## 5. query_id — Unique Per Invocation

In [ ]:
states = [initial_state(f"query {i}") for i in range(5)]
print("Each invocation gets a unique query_id:")
for s in states:
    print(f"  {s['query_id']}")

## 6. Memory Persistence Simulation

In [ ]:
# Simulate what LangGraph stores between turns
conversation_log = []

def simulate_turn(query: str, intent: str, summary: str, thread_id: str):
    entry = {"query": query, "summary": summary[:50], "intent": intent, "tickets": []}
    conversation_log.append(entry)
    return entry

turns = [
    ("What is GRR?", "knowledge_lookup", "GRR is 87.5%, above the 85% threshold"),
    ("Why did it drop in Q3?", "full_diagnostic", "Churn increased due to pricing changes"),
    ("Create a ticket for the GRR drop", "write_ticket", "Created DATA-042"),
]

for q, intent, summary in turns:
    simulate_turn(q, intent, summary, "thread-001")

print("Conversation history:")
for i, e in enumerate(conversation_log, 1):
    print(f"  Turn {i}: [{e['intent']}] {e['query'][:40]}")
    print(f"           → {e['summary'][:50]}")